### Data Pipeline - Data Ingestion to vector database

In [8]:
import os

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
### read all the pds inside the directory
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory."""

    all_documents = []

    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Ansh_Narayan_2026 (3).pdf
  ✓ Loaded 1 pages

Processing: Ansh_Narayan_Resume_Updated.pdf
  ✓ Loaded 1 pages

Processing: Nikita_Tyagi_Resume (6).pdf
  ✓ Loaded 1 pages

Total documents loaded: 3


In [11]:
all_pdf_documents


[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdfs\\Ansh_Narayan_2026 (3).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ansh_Narayan_2026 (3).pdf', 'file_type': 'pdf'}, page_content='ANSH NARAYAN \n+91 9389761875 | narayanansh94@gmail.com | LinkedIn \nPROFESSIONAL SUMMARY \nData Analyst with 1+ year of experience in SQL, Python, and Power BI, with hands -on exposure to AWS cloud services (EC2, S3, IAM, \nLambda) and Linux environments. Cleaned 70,000+ row datasets, built ETL pipelines using Pandas and n8n, and developed 3+ Powe r BI \ndashboards tracking sales, subscription, and operational KPIs. AWS Certified Cloud Practitioner (CLF -C02) with working knowledge of \nSQL objects (Stored Procedures, Views, CTEs, UDFs) across MySQL, PostgreSQL, and SQL Server.  \nTECHNICAL SKILLS \nCloud & OS: AWS (S3, IAM, Lambda, EC2) – Certified Cloud Practitioner, Linux (working familiarity), Windows \nProgramming: Python

In [12]:
### text spliting into chunks 

def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    """Split documents into smaller chunks."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"content: {split_docs[0].page_content[:200]}")  # Show first 500 characters of the first chunk
        print(f"metadata: {split_docs[0].metadata}")  # Show metadata of the first chunk
    return split_docs

In [14]:
chunks =  split_documents(all_pdf_documents)
chunks

split 3 documents into 15 chunks

Example chunk:
content: ANSH NARAYAN 
+91 9389761875 | narayanansh94@gmail.com | LinkedIn 
PROFESSIONAL SUMMARY 
Data Analyst with 1+ year of experience in SQL, Python, and Power BI, with hands -on exposure to AWS cloud serv
metadata: {'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdfs\\Ansh_Narayan_2026 (3).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ansh_Narayan_2026 (3).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdfs\\Ansh_Narayan_2026 (3).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ansh_Narayan_2026 (3).pdf', 'file_type': 'pdf'}, page_content='ANSH NARAYAN \n+91 9389761875 | narayanansh94@gmail.com | LinkedIn \nPROFESSIONAL SUMMARY \nData Analyst with 1+ year of experience in SQL, Python, and Power BI, with hands -on exposure to AWS cloud services (EC2, S3, IAM, \nLambda) and Linux environments. Cleaned 70,000+ row datasets, built ETL pipelines using Pandas and n8n, and developed 3+ Powe r BI \ndashboards tracking sales, subscription, and operational KPIs. AWS Certified Cloud Practitioner (CLF -C02) with working knowledge of \nSQL objects (Stored Procedures, Views, CTEs, UDFs) across MySQL, PostgreSQL, and SQL Server.  \nTECHNICAL SKILLS \nCloud & OS: AWS (S3, IAM, Lambda, EC2) – Certified Cloud Practitioner, Linux (working familiarity), Windows \nProgramming: Python

### Embedding And VectorDb

In [50]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [51]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape:
            (number_of_texts, embedding_dimension)
        """

        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 779.28it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\DELL\AppData\Local\Temp\ipykernel_19912\4106216118.py:30: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


VectorStore

In [52]:
import os
import uuid
from typing import List, Any

import numpy as np
import chromadb


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Initialize the vector store
vectorstore = VectorStore()

vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 45


In [53]:
chunks

[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdfs\\Ansh_Narayan_2026 (3).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Ansh_Narayan_2026 (3).pdf', 'file_type': 'pdf'}, page_content='ANSH NARAYAN \n+91 9389761875 | narayanansh94@gmail.com | LinkedIn \nPROFESSIONAL SUMMARY \nData Analyst with 1+ year of experience in SQL, Python, and Power BI, with hands -on exposure to AWS cloud services (EC2, S3, IAM, \nLambda) and Linux environments. Cleaned 70,000+ row datasets, built ETL pipelines using Pandas and n8n, and developed 3+ Powe r BI \ndashboards tracking sales, subscription, and operational KPIs. AWS Certified Cloud Practitioner (CLF -C02) with working knowledge of \nSQL objects (Stored Procedures, Views, CTEs, UDFs) across MySQL, PostgreSQL, and SQL Server.  \nTECHNICAL SKILLS \nCloud & OS: AWS (S3, IAM, Lambda, EC2) – Certified Cloud Practitioner, Linux (working familiarity), Windows \nProgramming: Python

In [54]:
#convert the text to embeddings

texts = [doc.page_content for doc in chunks]

#generate embeddings for the chunks

embeddings = embedding_manager.generate_embeddings(texts)

#store in the vector database 
vectorstore.add_documents(chunks , embeddings)


Generating embeddings for 15 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]


Generated embeddings with shape: (15, 384)
Adding 15 documents to vector store...
Successfully added 15 documents to vector store
Total documents in collection: 60


###Retriever Pipeline From VectorStore

In [56]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, "
            f"Score threshold: {score_threshold}"
        )

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize the RAG retriever
rag_retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager
)

rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [57]:
rag_retriever

In [58]:
rag_retriever.retrieve("What are the skills of Ansh Narayan?")

Retrieving documents for query: 'What are the skills of Ansh Narayan?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]


Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

Integration VectorDb Context Pipeline With LLM Output

In [63]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY was not found. Check the project-root .env file.")

llm = ChatGroq(
    api_key=api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=500,
)

# Simple RAG function to retrieve context and generate an answer
def rag_simple(query, retriever, llm, top_k=5):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(result["content"] for result in results) if results else ""
    if not context:
        return "No relevant documents found for the query."

    prompt = (
        "Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )
    response = llm.invoke(prompt)
    return response.content

In [68]:
answer = rag_simple("Ansh NArayan Email?", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'Ansh NArayan Email?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant documents found for the query.


Enhanced RAG Pipeline Features 

In [ ]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0.2,
    return_context=False
):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score,
      and optionally full context.
    """

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    # Prepare context and sources
    context = "\n\n".join(
        [doc["content"] for doc in results]
    )

    sources = [
        {
            "source": doc["metadata"].get(
                "source_file",
                doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:120] + "..."
        }
        for doc in results
    ]

    confidence = max(
        [doc["similarity_score"] for doc in results]
    )

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(
        prompt.format(
            context=context,
            query=query
        )
    )

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context

    return output


# Example usage
result = rag_advanced(
    "Ansh Narayan?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context Preview:", result["context"][:300])

Retrieving documents for query: 'Ansh Narayan profile summary ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
